# Ecommerce_Analytics: From Raw Data To Performance Analytics and Business Insights

The aim of this project is to design and implement an end-to-end ecommerce analytics pipeline by transforming and cleaning raw transactional CSV datasets from multiple sources. Followed by transforming the cleaned transactional data into a dimensional star schema, creating database and loading reliable data.

## Results
- A reliable database with clean data for easy and scalable revenue, product, and customer analysis. A curated dataset to export for Tableau visualization, and a dashboard with identified key metrics.

In [1]:
# Imports

import pandas as pd
import sqlite3
import os
from datetime import datetime
import logging
import sys
import hashlib

In [2]:
## Setup

# Directories
DATA_DIR = "./data/"
DB_DIR = "./database/"
DB_PATH = os.path.join(DB_DIR, "ecommerce.db")

# Ensure folders exist
os.makedirs(DB_DIR, exist_ok=True)

# logging function
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', handlers= [logging.FileHandler('ecommerce.log'), logging.StreamHandler()])

# Load data
orders = pd.read_csv(os.path.join(DATA_DIR, "orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(DATA_DIR, "order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(DATA_DIR, "customers_dataset.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "products_dataset.csv"))
payments = pd.read_csv(os.path.join(DATA_DIR, "order_payments_dataset.csv"))
reviews = pd.read_csv(os.path.join(DATA_DIR, "order_reviews_dataset.csv"))
category_name = pd.read_csv(os.path.join(DATA_DIR, "product_category_name_translation.csv"))

## Exploratory Data Assessment

Performing initial data profiling to:
- Confirm all the data was properly loaded.
- View samples of each dataset.
- Detect where null values exist.
- Check unique values in ID columns.
- Detect duplicate records.

In [3]:
# Quick EDA
for df, name in zip([orders, order_items, customers, products, payments, reviews, category_name],
                    ["orders","order_items","customers","products","payments","reviews", "category_name"]):
    print(f"\n\n\n{name.upper()}\n \nshape: {df.shape}")
    print(df.iloc[:, 0].nunique())
    print(df.info())
    print(df.head(2))




ORDERS
 
shape: (99441, 8)
99441
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purch

## Exploratory Data Assessment

More exploratory analysis to:
- Validate unique ID values, and whether they correspond across datasets.
- Identify null patterns
- Detect duplicate records
- Understand table cardinality prior to modeling

In [4]:
# Exploratory Data Assessment

# Check for orders that don't exist in order_items
orders_in_order_items = orders['order_id'].isin(order_items['order_id'])
print(orders_in_order_items.value_counts())

# Check order status of orders that dont exist in order_items
orders_wo_order_items = orders[~orders_in_order_items]
print(orders_wo_order_items.head())
print(orders_wo_order_items['order_status'].value_counts())

# Check for order customers that don't exist in customers data, and order item products that don't exist in products data
print(orders['customer_id'].isin(customers['customer_id']).value_counts())
print(order_items['product_id'].isin(products['product_id']).value_counts())

# Explore order_items, and how order_items_id relates to order_id to assess cardinality
print(order_items.describe())
order_items.sort_values(by='order_id').head(10)
order_items[order_items.duplicated(subset=['order_id'])].head(10)

# Explore null values in products
products[products['product_category_name'].isnull()]
products[products['product_weight_g'].isnull()]
products[(products['product_category_name'].isnull()) & (products['product_weight_g'].isnull())]
products[(products['product_name_lenght'].isnull()) & (products['product_category_name'].isnull())]

# Explore reviews dataset for nulls and duplicates
reviews[reviews.duplicated(subset=['review_id'])]
reviews.sort_values(by='review_id').head(50)

logger.info("Exploratory assessment completed!")

order_id
True     98666
False      775
Name: count, dtype: int64
                              order_id                       customer_id  \
266   8e24261a7e58791d10cb1bf9da94df5c  64a254d30eed42cd0e6c36dddb88adf0   
586   c272bcd21c287498b4883c7512019702  9582c5bbecc65eb568e2c1d839b5cba1   
687   37553832a3a89c9b2db59701c357ca67  7607cd563696c27ede287e515812d528   
737   d57e15fb07fd180f06ab3926b39edcd2  470b93b3f1cde85550fc74cd3a476c78   
1130  00b1cb0320190ca0daa2c88b35206009  3532ba38a3fd242259a514ac2b6ae6b6   

     order_status order_purchase_timestamp    order_approved_at  \
266   unavailable      2017-11-16 15:09:28  2017-11-16 15:26:57   
586   unavailable      2018-01-31 11:31:37  2018-01-31 14:23:50   
687   unavailable      2017-08-14 17:38:02  2017-08-17 00:15:18   
737   unavailable      2018-01-08 19:39:03  2018-01-09 07:26:08   
1130     canceled      2018-08-28 15:26:39                  NaN   

     order_delivered_carrier_date order_delivered_customer_date  \
266     

[2026-03-19 15:38:26,489] Exploratory assessment completed!


There are multiple orders that don't exist in the order items data. Further exploration shows that mjority of those orders were cancelled, unavilable, or otherwise. Custoomer IDs correspond across datasets, same with the product IDs. Order item IDs are assigned to individual items within the orders (not individual products). Reviews and order items datasets don't have columns with fully unique IDs that can identify each data entry, so a surrogate key will be created for those datasets. As the product category names are not in English, they will be replaced with their English translations for easy understanding by English speaking audience. While product ID's are consistent across datasets, some product category names are unavailable. Regarless, such rows still provide other necessary information, so a default value of 'unknown' will be assigned to null product category names. '1900-1-1 00:00:00' will also be assigned where date values are unavailable. 

## Data Transformation & Fact Table Definition

The primary fact table (fact_order_items) is defined at the order-item level. This will allow for product-level order revenue analysis. It will also preserve unit pricing and freight pricing components, and allow for aggregation where necessary at the order, product and category levels. Order items will be aggregated to order product level, where each product per order will be aggregated to reflect quantity sold.


In [5]:
# Aggregate to order-item level to establish correct fact table granularity
order_items[order_items.duplicated(subset=['order_id', 'product_id', 'seller_id', 'shipping_limit_date', 'freight_value'])] #order_items duplication on all columns except items and price
order_items = order_items.groupby(['order_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value'], as_index=False).agg({'order_item_id':'count'}).rename(columns={'order_item_id':'quantity', 'price':'unit_price', 'freight_value': 'unit_freight_value'}) # A
order_items['price'] = order_items['unit_price'] * order_items['quantity']
order_items['total_freight_value'] = order_items['unit_freight_value'] * order_items['quantity']

# Get English translation for product categories
products = products.merge(category_name, on='product_category_name', how='left')
products['product_category_name_english'] = products.apply(lambda row: row['product_category_name'] if pd.isna(row['product_category_name_english']) else row['product_category_name_english'], axis=1)
products = products.rename(columns={'product_name_lenght':'product_name_length', 'product_description_lenght':'product_description_length', 'product_category_name':'product_category_name_br', 'product_category_name_english':'product_category_name'})

# Assign default date replace null date values
na_date = pd.to_datetime('1900-1-1 00:00:00')
for column_name in ["order_purchase_timestamp", 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[column_name] = pd.to_datetime(orders[column_name])
    orders[column_name] = orders[column_name].fillna(na_date)

# Assign value to replace null product categories
products['product_category_name'] = products['product_category_name'].fillna('Unknown')
for column_name in ['product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[column_name] = products[column_name].fillna(0)

orders['order_date'] = pd.to_datetime(orders['order_purchase_timestamp'].dt.strftime('%Y%m%d'))
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])
reviews.review_comment_title = reviews.review_comment_title.fillna('No title provided')
reviews.review_comment_message = reviews.review_comment_message.fillna('No message provided')

## Surrogate Key Strategy

Surrogate keys are generated using MD5 hashing to ensure uniqueness across composite business keys. They will also simplify joins in the star schema, and support potential scalability.

In [6]:
def generate_surrogate_key(row, columns):
    # Combine values from chosen columns into a string
    column_string = ''.join(str(row[col]) for col in columns)
    # Generate hash
    return hashlib.md5(column_string.encode('utf-8')).hexdigest()

reviews['review_key'] = reviews.apply(lambda row: generate_surrogate_key(row, ['order_id', 'review_id']), axis=1)
order_items['order_item_key'] = order_items.apply(lambda row: generate_surrogate_key(row, ['order_id', 'product_id']), axis=1)
order_items = order_items.iloc[:, [9, 0, 1, 4, 6, 7, 5, 8, 2, 3]]

payments['payment_key'] = payments.apply(lambda row: generate_surrogate_key(row, ['order_id', 'payment_sequential']), axis=1)

# Insert default dimension members to preserve referential integrity
na_hash = '00000000000000000000000000000000'
products.loc[-1] = [na_hash, 'N/A', 0, 0, 0, 0, 0, 0, 0, 'N/A']
products.reset_index(inplace=True, drop=True)

customers.loc[-1] = [na_hash, na_hash, 0, 'N/A', 'N/A']
customers.reset_index(inplace=True, drop=True)

In [7]:
# Generate date dimension to enable time-series aggregation and filtering
dim_dates = pd.DataFrame({'timestamp': pd.date_range(orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max())})
dim_dates.loc[-1] = na_date
dim_dates.reset_index(inplace=True, drop=True)
dim_dates['date'] = pd.to_datetime(dim_dates['timestamp'].dt.strftime('%Y%m%d'))
dim_dates['year'] = dim_dates['timestamp'].dt.year
dim_dates['month'] = dim_dates['timestamp'].dt.month
dim_dates['day'] = dim_dates['timestamp'].dt.day
dim_dates['weekday'] = dim_dates['timestamp'].dt.day_name()

logger.info("Transform completed!")

[2026-03-19 15:38:48,304] Transform completed!


## Dimensional Modeling

A star schema is implemented with teh fact_order_items capturing measurable events, and other datsets as dimension tables storing descriptive attributes. Indexes are added on foreign keys to optimize join performance for downstream analytical queries and BI consumption.

In [8]:
# Connection & Constraints
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = OFF;") 
cursor = conn.cursor()

for table_name in ['fact_order_items', 'fact_payments', 'fact_reviews', 'dim_customers', 'dim_products', 'dim_date', 'dim_orders']:
    cursor.execute(f"""
    DROP TABLE IF EXISTS {table_name}
    """)

### CREATE TABLES
conn.execute("PRAGMA foreign_keys = ON;")

# Dimensions
cursor.execute("""
CREATE TABLE dim_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_date DATETIME,
    order_status TEXT,
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME
);
""")

cursor.execute("""
CREATE TABLE dim_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state TEXT
);
""")

cursor.execute("""
CREATE TABLE dim_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length REAL,
    product_description_length REAL,
    product_photos_qty REAL,
    product_weight_g REAL,
    product_length_cm REAL,
    product_height_cm REAL,
    product_width_cm REAL,
    product_category_name_br TEXT
);
""")

cursor.execute("""
CREATE TABLE dim_date (
    date DATETIME PRIMARY KEY,
    timestamp DATETIME,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    weekday TEXT
);
""")

# Facts
cursor.execute("""
CREATE TABLE fact_order_items (
    order_item_key TEXT PRIMARY KEY,
    order_id TEXT,
    product_id TEXT,
    unit_price REAL,
    quantity INTEGER,
    price REAL,
    unit_freight_value REAL,
    total_freight_value REAL,
    seller_id TEXT, 
    shipping_limit_date TEXT,
    FOREIGN KEY (order_id) REFERENCES dim_orders (order_id) ON DELETE CASCADE,
    FOREIGN KEY (product_id) REFERENCES dim_products (product_id) ON DELETE CASCADE
);
""")

# Payments
cursor.execute("""
CREATE TABLE fact_payments (
    payment_key TEXT PRIMARY KEY,
    order_id TEXT,
    payment_sequential INTEGER,
    payment_type TEXT,
    payment_installments INTEGER,
    payment_value REAL,
    FOREIGN KEY (order_id) REFERENCES dim_orders (order_id) ON DELETE CASCADE
);
""")

# Reviews
cursor.execute("""
CREATE TABLE fact_reviews (
    review_key TEXT PRIMARY KEY,
    order_id TEXT,
    review_id TEXT,
    review_score INTEGER,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME,
    FOREIGN KEY (order_id) REFERENCES dim_orders (order_id) ON DELETE CASCADE
);
""")

### PERFORMANCE INDEXES
# index order_id on all tables to make "Master View" joins instant.
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_order ON fact_order_items (order_id);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_pay_order ON fact_payments (order_id);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_order ON fact_reviews (order_id);")

conn.close()

In [9]:
## Load into SQLite (Star Schema)

conn = sqlite3.connect(DB_PATH)

def load_data (df, table_name):
    df.to_sql(table_name, conn, if_exists='append', index=False)

### Dimension tables
load_data(orders, "dim_orders")
load_data(customers, "dim_customers")
load_data(products, "dim_products")
load_data(dim_dates, "dim_dates")

### Add Fact tables
# Orders
load_data(order_items, "fact_order_items")
# Payments
load_data(payments, "fact_payments")
# Reviews
load_data(reviews, "fact_reviews")

conn.close()
logger.info("Star schema created successfully!")

[2026-03-19 15:39:19,424] Star schema created successfully!


## Analytical Queries

Calculate core business KPIs using queries:
- Monthly revenue trends
- Month Over Month Growth
- Avg. Revenue Per Order
- Top Products
- Top Customers

In [10]:
## Analytics Queries

# Connect again to SQLite:
conn = sqlite3.connect(DB_PATH)

#  Query function
def run_sql(query):
    return pd.read_sql(query, conn)

# Monthly Revenue
query = """
SELECT strftime('%Y%m', d.order_date) AS month,
       SUM(price) AS revenue
FROM fact_order_items f
join dim_orders d on f.order_id = d.order_id
GROUP BY month
ORDER BY month;
"""
monthly_revenue = run_sql(query)
monthly_revenue

,month,revenue
0,201609,267.36
1,201610,49507.66
2,201612,10.90
3,201701,120312.87
4,201702,247303.02
5,201703,374344.30
6,201704,359927.23
7,201705,506071.14
8,201706,433038.60
9,201707,498031.48


In [11]:
# Month-over-Month Growth

query = """
WITH monthly_revenue AS (
    SELECT strftime('%Y%m', d.order_date) AS month,
    SUM(price) AS revenue
    FROM fact_order_items f
    join dim_orders d on f.order_id = d.order_id
    GROUP BY month
)
SELECT month,
       revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month))*100.0 / LAG(revenue) OVER (ORDER BY month), 2) AS mom_growth_pct
FROM monthly_revenue;
"""
mom_growth = run_sql(query)
mom_growth

,month,revenue,prev_month_revenue,mom_growth_pct
0,201609,267.36,NaN,NaN
1,201610,49507.66,267.36,18417.23
2,201612,10.90,49507.66,-99.98
3,201701,120312.87,10.90,1103687.80
4,201702,247303.02,120312.87,105.55
5,201703,374344.30,247303.02,51.37
6,201704,359927.23,374344.30,-3.85
7,201705,506071.14,359927.23,40.60
8,201706,433038.60,506071.14,-14.43
9,201707,498031.48,433038.60,15.01


In [12]:
# Top Product Categories

query = """
SELECT p.product_category_name,
    SUM(price) AS total_revenue,
    COUNT(DISTINCT(f.order_id)) AS num_orders
FROM fact_order_items f
JOIN dim_products p
ON f.product_id = p.product_id
GROUP BY product_category_name
ORDER BY total_revenue DESC
LIMIT 20;
"""
top_categories = run_sql(query)
top_categories

,product_category_name,total_revenue,num_orders
0,health_beauty,1258681.34,8836
1,watches_gifts,1205005.68,5624
2,bed_bath_table,1036988.68,9417
3,sports_leisure,988048.97,7720
4,computers_accessories,911954.32,6689
5,furniture_decor,729762.49,6449
6,cool_stuff,635290.85,3632
7,housewares,632248.66,5884
8,auto,592720.11,3897
9,garden_tools,485256.46,3518


In [13]:
### Average Revenue per Order by Product Category

query = """
SELECT p.product_category_name,
       ROUND(AVG(f.price),2) AS avg_order_rev,
       COUNT(DISTINCT(f.order_id)) AS num_orders
FROM fact_order_items f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY product_category_name
ORDER BY avg_order_rev DESC;
"""
avg_order_value_by_category = run_sql(query)
avg_order_value_by_category

,product_category_name,avg_order_rev,num_orders
0,computers,1231.84,181
1,small_appliances_home_oven_and_coffee,624.29,75
2,home_appliances_2,484.26,234
3,agro_industry_and_commerce,396.34,182
4,small_appliances,301.66,630
...,...,...,...
69,electronics,62.47,2550
70,cds_dvds_musicals,60.83,12
71,diapers_and_hygiene,58.06,27
72,flowers,38.28,29


In [14]:
### Top Customers by Lifetime Value**

query = """
SELECT o.customer_id,
       COUNT(DISTINCT(f.order_id)) AS total_orders,
       SUM(f.price) AS total_spent
FROM fact_order_items f
JOIN dim_orders o ON f.order_id = o.order_id
GROUP BY o.customer_id
ORDER BY total_spent DESC
LIMIT 20;
"""
top_customers = run_sql(query)
top_customers

,customer_id,total_orders,total_spent
0,1617b1357756262bfa56ab541c47bc16,1,13440.00
1,ec5b2ba62e574342386871631fafd3fc,1,7160.00
2,c6e2731c5b391845f6800c97401a43a9,1,6735.00
3,f48d464a0baaea338cb25f816991ab1f,1,6729.00
4,3fd6777bbce08a352fddd04e4a7cc8f6,1,6499.00
5,05455dfa7cd02f13d132aa7a6a9729c6,1,5934.60
6,df55c14d1476a9a3467f131269c2477f,1,4799.00
7,24bbf5fd2f2e1b359ee7de94defc4a15,1,4690.00
8,e0a2412720e9ea4f26c1ac985f6a7358,1,4599.90
9,3d979689f636322c62418b6346b1c6d2,1,4590.00


In [15]:
conn.close()

## Tableau Export Layer

A consolidated analytical view is created to simplify Tableau data modeling, minimize dashboard-side transformations, and standardize KPI definitions across reports.

This view serves as a data source for Tableau dashboards.

In [16]:
# EXPORT DATA FOR TABLEAU
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
DROP VIEW  IF EXISTS v_tableau_orders;
""")

cursor.execute("""
CREATE VIEW IF NOT EXISTS v_tableau_orders AS
SELECT 
    COALESCE(f.order_item_key, '00000000000000000000000000000000')AS order_item_key,
    COALESCE(o.order_id, '00000000000000000000000000000000') AS order_id,
    COALESCE(f.product_id, '00000000000000000000000000000000') AS product_id,
    COALESCE(f.unit_price, 0.0) AS unit_price,
    COALESCE(f.quantity, 0) AS quantity,
    COALESCE(f.price, 0.0) AS price,
    COALESCE(f.unit_freight_value, 0.0) AS unit_freight_value,
    COALESCE(f.total_freight_value, 0.0) AS total_freight_value,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.order_approved_at,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,
    COALESCE(p.product_category_name, 'N/A') AS product_category_name
FROM dim_orders o
LEFT JOIN fact_order_items f ON o.order_id = f.order_id
LEFT JOIN dim_customers c ON o.customer_id = c.customer_id
LEFT JOIN dim_products p ON f.product_id = p.product_id;
""")


In [17]:
# Pull data from view
run_sql('''Select * from v_tableau_orders LIMIT 10''')

,order_item_key,order_id,product_id,unit_price,quantity,price,unit_freight_value,total_freight_value,customer_id,order_date,order_status,order_approved_at,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name
0,9345db19a82b1c08b8c592ded81298d8,e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,29.99,1,29.99,8.72,8.72,9ef432eb6251297304e76186b10a928d,2017-10-02 00:00:00,delivered,2017-10-02 11:07:15,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,housewares
1,6811ca58135f48361d16fe2f0c4bdcf2,53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,118.70,1,118.70,22.76,22.76,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 00:00:00,delivered,2018-07-26 03:24:27,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,perfumery
2,1539453e60cbcd5268b43916c14f18be,47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,159.90,1,159.90,19.22,19.22,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 00:00:00,delivered,2018-08-08 08:55:23,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,auto
3,0f085eba1f6d94b9a8c4e1fe5c8713d5,949d5b44dbf5de918fe9c16f97b45f8a,d0b61bfb1de832b15ba9d266ca96e5b0,45.00,1,45.00,27.20,27.20,f88197465ea7920adcdbec7375364d82,2017-11-18 00:00:00,delivered,2017-11-18 19:45:59,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,pet_shop
4,02df4c0fee1656381505b2f098e4a881,ad21c59c0840e6cb83a9ceb5573f8159,65266b2da20d04dbe00c5c2d3bb7859e,19.90,1,19.90,8.72,8.72,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 00:00:00,delivered,2018-02-13 22:20:29,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,stationery
5,63592a24519e86ffd6eff536baa1f52f,a4591c265e18cb1dcee52889e2d8acc3,060cb19345d90064d1015407193c233d,147.90,1,147.90,27.36,27.36,503740e9ca751ccdda7ba28e9ab8f608,2017-07-09 00:00:00,delivered,2017-07-09 22:10:13,80bb27c7c16e8f973207a5086ab329e2,86320,congonhinhas,PR,auto
6,25d2c98cb0a83343e0a288cf490da148,136cce7faa42fdb2cefd53fdc79a6098,a1804276d9941ac0733cfd409f5206eb,49.90,1,49.90,16.05,16.05,ed0271e0b7da060a393796590e7b737a,2017-04-11 00:00:00,invoiced,2017-04-13 13:25:17,36edbb3fb164b1f16485364b6fb04c73,98900,santa rosa,RS,Unknown
7,147e4a15ce15aa4049b470c2251b8f70,6514b8ad8028c9f2cc2374ded245783f,4520766ec412348b8d4caa5e8a18c464,59.99,1,59.99,15.17,15.17,9bdf08b4b3b52b5526ff42d37d47f222,2017-05-16 00:00:00,delivered,2017-05-16 13:22:11,932afa1e708222e5821dac9cd5db4cae,26525,nilopolis,RJ,auto
8,9bbdd48b5bc3e9ccb1a1664a4792e0cb,76c6e866289321a7c93b82b54852dc33,ac1789e492dcd698c5c10b97a671243a,19.90,1,19.90,16.05,16.05,f54a9f0e6b351c431402b8461ea51999,2017-01-23 00:00:00,delivered,2017-01-25 02:50:47,39382392765b6dc74812866ee5ee92a7,99655,faxinalzinho,RS,furniture_decor
9,14dfbd597b9d8f41e0b9d753d7f98220,e69bfb5eb88e0ed6a785585b27e16dbf,9a78fb9862b10749a117f7fc3c31f051,149.99,1,149.99,19.77,19.77,31ad1d1b63eb9962463f764d4e6e0c9d,2017-07-29 00:00:00,delivered,2017-07-29 12:05:32,299905e3934e9e181bfb2e164dd4b4f8,18075,sorocaba,SP,office_furniture


In [18]:
run_sql('''Select SUM(price) as total_revenue from v_tableau_orders''')

,total_revenue
0,13591643.7


In [19]:
# Save data to file
df_tableau = pd.read_sql_query("SELECT * FROM v_tableau_orders", conn)

# Export to CSV for Tableau
df_tableau.to_csv('orders_tableau.csv', index=False)

conn.close()
logger.info("Export Complete: orders_tableau.csv is ready for visualization.")

[2026-03-19 15:39:54,913] Export Complete: orders_tableau.csv is ready for visualization.
